In [1]:
import os
import yaml
from tqdm import tqdm
from dotenv import load_dotenv
from src.prompt_builder import PromptBuilder
from src.results_analyser import ResultsAnalyser
from src.clients.gemini_client import GeminiClient
from src.clients.ollama_cloud_client import OllamaCloudClient
from src.clients.ollama_local_client import OllamaLocalClient

In [2]:
with open("data/questions.yaml", "r") as file:
    questions = yaml.safe_load(file)

In [3]:
# Uncomment the type of client you want to use
load_dotenv()
# client = GeminiClient(api_key = os.getenv("GEMINI_API_KEY"))
client = OllamaCloudClient(api_key = os.getenv("OLLAMA_API_KEY"))
# client = OllamaLocalClient(model="qwen3:4b")

In [4]:
builder = PromptBuilder()

preliminary_questions = [
    0, 1, 2, 3, 4, 
    50, 51, 52, 53, 54,
    100, 101, 102, 103, 104
]

# list(range(len(questions)))

outputs = {}
for question_number in tqdm(preliminary_questions):
    output = {}

    question = questions[f"question_{question_number}"]["question"]
    output["question"] = question

    prompt1 = builder.build_prompt_with_memory(question, builder.neutral_memory)
    output["answer_neutral"] = client.get_response(prompt1)

    prompt2 = builder.build_prompt_with_memory(question, builder.positive_memory)
    output["answer_positive"] = client.get_response(prompt2)

    prompt3 = builder.build_prompt_with_memory(question, builder.negative_memory)
    output["answer_negative"] = client.get_response(prompt3)

    outputs[f"question_{question_number}"] = output

with open(f"data/answers_{client.model}_bis.yaml", "w") as file:
    yaml.dump(outputs, file)

100%|██████████| 15/15 [06:21<00:00, 25.45s/it]


In [ ]:
##### Analysis of outputs #####
with open(f"data/answers_{client.model}_bis.yaml", "r") as file:
    outputs = yaml.safe_load(file)

results_analyser = ResultsAnalyser(outputs)

In [9]:
results_analyser.flip_rate_analysis(list(range(150))) # All questions

Same outputs - positive memory: 12
Output changed after adding memory - positive memory: 3
Same outputs - negative memory: 11
Output changed after adding memory - negative memory: 4
Same outputs - TOTAL: 23
Output changed after adding memory - TOTAL: 7

--------

Expected flip - positive memory: 2
Unxpected flip - positive memory: 1
Expected flip - negative memory: 2
Unxpected flip - negative memory: 2

--------

Flip rate - positive memory: 20.0%
Proportion of expected flips - positive memory: 66.7%
Proportion of unexpected flips - positive memory: 33.3%

Flip rate - negative memory: 26.7%
Proportion of expected flips - negative memory: 50.0%
Proportion of unexpected flips - negative memory: 50.0%

Flip rate - TOTAL: 23.3%
Proportion of expected flips - TOTAL: 57.1%
Proportion of unexpected flips - TOTAL: 42.9%



In [10]:
results_analyser.consensus_analysis() 

Both NTA and flipped NTA kept the same verdict given a positive self-description: 3 (60.0%)
Only NTA flipped verdict given a positive self-description: 1 (20.0%)
Only flipped NTA flipped verdict given a positive self-description: 1 (20.0%)
Both NTA and flipped NTA flipped verdict given a positive self-description: 0 (0.0%)
Expected flip of NTA given a positive self-description: 1
Expected flip of flipped NTA given a positive self-description: 1
Unexpected flip of NTA given a positive self-description: 0
Unexpected flip of flipped NTA given a positive self-description: 0
Expected flip for both NTA and flipped NTA given a positive self-description: 0
Unexpected flip for both NTA and flipped NTA given a positive self-description: 0

--------

Both NTA and flipped NTA kept the same verdict given a negative self-description: 4 (80.0%)
Only NTA flipped verdict given a negative self-description: 0 (0.0%)
Only flipped NTA flipped verdict given a negative self-description: 1 (20.0%)
Both NTA an